[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# JSON in a Response &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `network` and `by_id`.
Run it first.


In [1]:
import importlib
import json
import sys
import urllib.request
from datetime import date
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()

network = requests.get(f"{BASE}/network", timeout=10).json()
by_id = {station["id"]: station for station in network["stations"]}

print("ready:", BASE)
print("Open-Meteo:", OPEN_METEO)


ready: http://127.0.0.1:8765
Open-Meteo: https://archive-api.open-meteo.com/v1/archive


**1.** Two members, and a count across every station.


In [2]:
instruments = sum(len(station["instruments"]) for station in network["stations"])

print(network["name"], "|", network["updated"], "|", instruments, "instruments")


Practice API station network | 2026-03-01T09:00:00Z | 9 instruments


`sum` over a generator adds up the length of every station's list of instruments.


**2.** One station, laid out.


In [3]:
print(json.dumps(by_id["tromso"], indent=2))


{
  "id": "tromso",
  "name": "Tromso",
  "location": {
    "latitude": 69.65,
    "longitude": 18.96,
    "elevation_m": 100
  },
  "instruments": [
    {
      "kind": "thermometer",
      "installed": "2019-05-01",
      "last_calibrated": "2026-01-20"
    },
    {
      "kind": "anemometer",
      "installed": "2019-05-01",
      "last_calibrated": "2025-02-11"
    }
  ],
  "status": null
}


Tromso's `status` is written `null`, because `json.dumps` writes Python's `None` back as JSON's
`null`.


**3.** A list inside every item of a list.


In [4]:
for station in network["stations"]:
    for instrument in station["instruments"]:
        if instrument["last_calibrated"] is None:
            print(f"{station['name']}: {instrument['kind']}")


Oslo: rain gauge
Svalbard: rain gauge


Every instrument has a `last_calibrated` key, so indexing it is safe. Only its value can be `None`.


**4.** A field that is missing, found with `not in`.


In [5]:
print([station["id"] for station in network["stations"] if "elevation_m" not in station["location"]])


['svalbard']


`not in` tests for the key itself, which is the right test here: the field is missing, not `null`.


**5.** Columns to rows, and the coldest night.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [6]:
weather = requests.get(OPEN_METEO, timeout=30, params={
    "latitude": 60.39, "longitude": 5.32, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum", "models": "era5"}).json()

daily = weather["daily"]
rows = [dict(zip(daily, row)) for row in zip(*daily.values())]
coldest = min(rows, key=lambda row: row["temperature_2m_min"])

print(coldest["time"], coldest["temperature_2m_min"], weather["daily_units"]["temperature_2m_min"])


2025-01-17 6.4 °C


`min` with a `key` compares the rows by one field and returns the whole row, so the date comes with
the temperature.


**6.** Dates compared, across a list inside a list.


In [7]:
def installed_before(network, year):
    """A (station id, instrument kind) pair for every instrument installed before the year began."""
    first_day = date(year, 1, 1)
    return [(station["id"], instrument["kind"])
            for station in network["stations"]
            for instrument in station["instruments"]
            if date.fromisoformat(instrument["installed"]) < first_day]


print(installed_before(network, 2019))


[('bergen', 'thermometer'), ('bergen', 'rain gauge'), ('oslo', 'thermometer'), ('oslo', 'rain gauge')]


A comprehension with two `for` clauses walks a list inside a list, in the same order as the nested
loops of task 3. Parsed dates compare as dates. Text in exactly this ISO 8601 form would happen to
compare correctly too, but a date written any other way would not, and `date.fromisoformat` raises
for it instead.


---

&#8592; **Back to:** [JSON in a Response](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/06-json-in-a-response.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
